In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
import numpy as np

In [ ]:
# Representative cluster from data_processing.ipynb
# Distance: median network distance to CAI proxy
# Density: median ACS-derived housing density
# Terrain: multiplier derived from p75 road grade
clusters = {
    "Urban Honolulu": {
        "distance_miles": 3.72,
        "m_terrain": 1.2,
        "m_deployment": 1.1,
        "density": 2248.88
    },
    "Suburban Mililani": {
        "distance_miles": 3.46,
        "m_terrain": 1.2,
        "m_deployment": 1.0,
        "density": 979.78
    },
    "Rural Waianae": {
        "distance_miles": 4.37,
        "m_terrain": 1.0,
        "m_deployment": 1.1,
        "density": 567.74
    },
    "Remote Ocean View": {
        "distance_miles": 4.49,
        "m_terrain": 1.5,
        "m_deployment": 1.3,
        "density": 22.90
    }
}
#baseline model assumptions

#aerial cost per mile
c_mile_aerial = 42000
#drop cost per household
c_drop = 1200  

#LEO satellite hardware threshold
satellite_upfront = 600
#Multi year total lifecycle threshold
satellite_lifecycle = 7800   

In [ ]:
def calculate_fiber_cost(
    d_miles,
    c_mile,
    m_terrain,
    m_deployment,
    housing_density,
    service_area_km2,
    c_drop,
    years=10,
    maintenance_rate=0.02
):
    households_served = housing_density * service_area_km2    
    #if cluster has zero density
    if households_served <= 0:
        households_served =1
        
    total_backbone=(d_miles * c_mile * m_terrain * m_deployment)
    capex_cost_per_hh = (total_backbone / households_served) + c_drop
    lifecycle_cost_per_hh = capex_cost_per_hh * (1+ (maintenance_rate * years))

    return lifecycle_cost_per_hh, capex_cost_per_hh, households_served

In [ ]:
#optimal deployment choice 
results=[]
for name, data in clusters.items():
    lifecycle_cost, capex_cost_per_hh, households_served = calculate_fiber_cost(
    d_miles=data["distance_miles"],
    c_mile=c_mile_aerial,
    m_terrain=data["m_terrain"],
    m_deployment=data["m_deployment"],
    housing_density=data["density"],
    service_area_km2=1,
    c_drop=c_drop,
)
    preferred_tech = "Fiber" if lifecycle_cost <= satellite_lifecycle else "Satellite"
    results.append({
    "Cluster": name,
    "Density (hh/km²)": data["density"],
    "Fiber CAPEX / HH ($)": round(capex_cost_per_hh, 2),
    "Fiber Lifecycle / HH ($)": round(lifecycle_cost, 2),
    "Optimal Technology": preferred_tech
})
df_results = pd.DataFrame(results)
print("Baseline Cost Model Results")
display(df_results)

In [ ]:
#generate fiber cost across a range of housing densitites
def density_threshold_analysis(
    cluster_name,
    cluster_data,
    density_values,
    c_mile,
    c_drop,
    satellite_lifecycle,
    service_area_km2=1
):
    density_results = []

    #run density sweep
    for density in density_values:
        lifecycle_cost, fiber_cost, households_served = calculate_fiber_cost(
            d_miles=cluster_data["distance_miles"],
            c_mile=c_mile,
            m_terrain=cluster_data["m_terrain"],
            m_deployment=cluster_data["m_deployment"],
            housing_density=density,
            service_area_km2=service_area_km2,
            c_drop=c_drop
        )
        technology = (
            "Fiber"
            if lifecycle_cost <= satellite_lifecycle
            else "Satellite"
        )
        density_results.append({
            "Cluster": cluster_name,
            "Density (hh/km²)": density,
            "Households Served": households_served,
            "Fiber Lifecycle Cost ($/HH)": lifecycle_cost,
            "Technology": technology
        })
    threshold = find_crossover_density(
        cluster_data=cluster_data,
        density_values=density_values,
        c_mile=c_mile,
        c_drop=c_drop,
        satellite_lifecycle=satellite_lifecycle,
        service_area_km2=service_area_km2
    )
    return pd.DataFrame(density_results), threshold

In [ ]:
 #define function to find housing density where fiber lifecycle=satellite
def find_crossover_density(
    cluster_data,
    density_values,
    c_mile,
    c_drop,
    satellite_lifecycle,
    service_area_km2=1
    ):
    def lifecycle_cost_difference(density):
        lifecycle_cost, _, _ = calculate_fiber_cost(
            d_miles=cluster_data["distance_miles"],
            c_mile=c_mile,
            m_terrain=cluster_data["m_terrain"],
            m_deployment=cluster_data["m_deployment"],
            housing_density=density,
            service_area_km2=service_area_km2,
            c_drop=c_drop
        )
        return lifecycle_cost - satellite_lifecycle
   #Brent's method to solve for crossover density
    try:
        threshold = brentq(
            lifecycle_cost_difference,
            min(density_values),
            max(density_values)
        )

    except ValueError:
        threshold = None
    return threshold

In [ ]:
density_values = [5,10,25,50,100,250,500,1000,2500]
all_density_results = []
#run density sweep for every cluster
for name, data in clusters.items():
    density_df, threshold = density_threshold_analysis(
        cluster_name=name,
        cluster_data=data,
        density_values=density_values,
        c_mile=c_mile_aerial,
        c_drop=c_drop,
        satellite_lifecycle=satellite_lifecycle
    )
    all_density_results.append(density_df)
#combine into one datafram
density_df = pd.concat(all_density_results, ignore_index=True)

In [ ]:
#plot fiber lifecycle cost vs housing density per cluster
plt.figure(figsize=(8,5))
for cluster in density_df["Cluster"].unique():
    subset = density_df[density_df["Cluster"] == cluster]

    plt.plot(
        subset["Density (hh/km²)"],
        subset["Fiber Lifecycle Cost ($/HH)"],
        marker="o",
        label=cluster
    )
#reference line
plt.axhline(
    satellite_lifecycle,
    linestyle="--",
    color="black",
    label="Satellite Lifecycle Cost"
)

plt.xlabel("Housing Density (households/km²)")
plt.ylabel("Fiber Lifecycle Cost ($/household)")
plt.title("Fiber Lifecycle Cost vs. Housing Density by Hawaii Deployment Environment")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
#solve crossover density for each cluster
threshold_results = []

for name, data in clusters.items():

    density_df, threshold = density_threshold_analysis(
        cluster_name=name,
        cluster_data=data,
        density_values=density_values,
        c_mile=c_mile_aerial,
        c_drop=c_drop,
        satellite_lifecycle=satellite_lifecycle
    )

    threshold_results.append({"Cluster": name,"Crossover Density (hh/km²)": threshold})

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df)

In [ ]:
#find how crossover density shifts as fiber construction cost changes
construction_sensitivity = []
c_mile_values = [20000,42000,60000,80000]
for c_mile in c_mile_values:
    for name,data in clusters.items():
        density_df, threshold = density_threshold_analysis(
            cluster_name=name,
            cluster_data=data,
            density_values=density_values,
            c_mile=c_mile,
            c_drop=c_drop,
            satellite_lifecycle=satellite_lifecycle
        )

        construction_sensitivity.append({
            "Construction Cost ($/mile)": c_mile,
            "Cluster": name,
            "Crossover Density (hh/km²)": threshold
        })

construction_df = pd.DataFrame(construction_sensitivity)
display(construction_df)

In [ ]:
#plot crossover density vs.construction cost per mile for each cluster
for cluster in construction_df["Cluster"].unique():
    subset = construction_df[construction_df["Cluster"] == cluster]

    plt.plot(
        subset["Construction Cost ($/mile)"],
        subset["Crossover Density (hh/km²)"],
        marker="o",
        label=cluster
    )

plt.xlabel("Fiber Construction Cost ($/mile)")
plt.ylabel("Required Density for Fiber Deployment (hh/km²)")
plt.title( "Construction Cost Sensitivity of Fiber Deployment Threshold")

plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def monte_carlo_crossover_density(cluster_data,simulations=1000, satellite_lifecycle=7800, seed=42):
    np.random.seed(seed)
    thresholds = []
    failed=0

    for i in range(simulations):
        #randomly sample uncertain parameters
        c_mile = np.random.uniform(20000, 80000)
        c_drop_random = np.random.uniform(800,2000)     
        terrain_multiplier = cluster_data["m_terrain"] * np.random.uniform( 0.8,1.2)
        deployment_multiplier = cluster_data["m_deployment"] * np.random.uniform(0.8,1.2)

        def lifecycle_cost_difference(density):
            lifecycle_cost, _, _ = calculate_fiber_cost(
                d_miles=cluster_data["distance_miles"],
                c_mile=c_mile,
                m_terrain=terrain_multiplier,
                m_deployment=deployment_multiplier,
                housing_density=density,
                service_area_km2=1,
                c_drop=c_drop_random
            )
            return lifecycle_cost - satellite_lifecycle

        #search for crossover
        try:
            threshold = brentq(lifecycle_cost_difference,1,5000)
            thresholds.append(threshold)
        except ValueError:
            failed += 1
    if failed:
        print(f"{failed} of {simulations} had no crossover in [1, 5000] and were skipped)")
    return pd.Series(thresholds)

In [ ]:
#running monte carlo simulation for every cluster
monte_carlo_results = {}
for name, data in clusters.items():
    results = monte_carlo_crossover_density(
        data,
        simulations=1000
    )
    monte_carlo_results[name] = results
    print(name)
    print(results.describe())
    print()

In [ ]:
#summarize each cluster's monte carlo distribution
summary = []
for cluster, values in monte_carlo_results.items():
    summary.append({
        "Cluster": cluster,
        "Mean Crossover Density": values.mean(),
        "5th Percentile": values.quantile(0.05),
        "95th Percentile": values.quantile(0.95)
    })
display(pd.DataFrame(summary))

In [ ]:
#histogram of crossover density per cluster
for name, results in monte_carlo_results.items():
    plt.figure(figsize=(8,5))

    plt.hist(results,bins=40)

    plt.axvline(
        results.mean(),
        linestyle="--",
        label=f"Mean:{results.mean():.1f}"
    )

    plt.xlabel("Crossover Density (households/km²)")
    plt.ylabel("Simulation Count")
    plt.title(f"Monte Carlo Distribution of Crossover Density: {name}")

    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
#boxplot comparison of crossover density uncertainty across all clusters
plt.figure(figsize=(9,5))
plt.boxplot(
    list(monte_carlo_results.values()), 
    tick_labels=list(monte_carlo_results.keys())
)
plt.ylabel("Crossover Density (households/km²)")
plt.title("Monte Carlo Uncertainty in Fiber-Satellite Economic Threshold")
plt.grid(True)
plt.show()